# Notebook 02 — Preprocessing and Physics-Informed Feature Engineering

## RustWeatherML · PhD Team (Physics · Mathematics · Machine Learning)

This notebook is **not** a cosmetic rewrite of the original. Every step
here is driven by a concrete finding from the Notebook 01 EDA:

| EDA finding (Nb01) | Action taken here |
|---|---|
| `windspeed` skew = +1.26 | `log(1+|U|)` reduces skewness |
| `precipitation` skew = +11, kurt = +174 | zero-inflated → `log1p`; target looks at the future |
| `cloudcover` is bimodal | `is_overcast` flag ($\ge 87.5$%) |
| $S^{\downarrow}$ is bimodal day/night | replace with the **clearness index** $\tau = S/S_{clear}$ |
| $\rho(T, T_{ap}) = 0.989$ | **drop** `apparent_temperature` (redundant) |
| $\rho(|U|, G) = 0.932$ | keep $|U|$, derive `gust_excess = G - |U|` |
| $\rho(S, N) = -0.16$ is weak | the clearness index isolates the cloud effect |
| $\rho(1)\approx 0.99$, $\rho(24)>0.83$ universal | lags $\{1,3,6,12,24,48\}$ |
| 2 violations $G < |U|$ (rounding) | `clip(G \to \max(G,|U|))` |
| Buggy `day_of_week` formula | use `chrono::NaiveDate` |
| `day_of_year` ignored leap years | `NaiveDate.ordinal()` |
| Forward-fill crossed city boundaries | `.fill_null(...).over("city")` |
| Target `will_rain = precip_now>0` is **leakage** | `will_rain_next_24h` via future window |

### Physical features produced below

#### Thermodynamics (Magnus-Tetens, Bolton 1980)
1. $e_s(T) = 6.112 \exp\!\left(\frac{17.67\,T}{T+243.5}\right)$ — saturation vapor pressure
2. $e = e_s(T_d)$ — actual vapor pressure
3. $VPD = e_s(T) - e$ — vapor-pressure deficit
4. $r = 0.622\,e/(P_{msl} - e)$ — mixing ratio
5. $q = r/(1+r)$ — specific humidity

#### Stability
6. $\Delta T_d = T - T_d$ — dewpoint depression
7. $z_{LCL} \approx 125 \cdot \Delta T_d$ — lifting condensation level proxy

#### Vector kinematics
8. $u = -|U|\sin(\theta\pi/180)$ — zonal component
9. $v = -|U|\cos(\theta\pi/180)$ — meridional component
10. `gust_excess` = $G - |U|$ — residual turbulence

#### Solar geometry
11. $\cos(z) = \sin\phi\sin\delta + \cos\phi\cos\delta\cos H$
12. $L_{day}(\phi, doy)$ — day length
13. $\tau = S^{\downarrow}/S_{clear}$ — clearness index

Notation: $\phi$=lat, $\delta$=solar declination, $H$=hour angle.

#### Cyclic time encoding (bug-fixed)
14–17. `hour`, `dow`, `month`, `doy` as sin/cos pairs

#### Lags (motivated by the Nb01 ACF)
18–25. $T$, $P$, $T_d$, $|U|$, `precip` over $\{1, 3, 6, 12, 24, 48\}$ h

#### Rolling windows (24 h)
26–32. mean / std / min / max / range

#### Gradients
33–37. $\Delta T$, $\Delta P$, $\Delta T_d$, $\Delta RH$, $\Delta |U|$ over $\{1, 3, 6\}$ h

#### Physical interactions
38–40. $T \cdot RH$, $|U| \cdot \Delta T_d$, $|U| \cdot VPD$

#### Geographical
41. $|\phi|$ (distance from the equator)
42–49. *one-hot* of the 8 Köppen zones present in Nb01

#### Leakage-free targets
50. `temp_next_{24,48,72}h` (already in Nb01)
51. `will_rain_next_24h` = $\max_{1\le k \le 24}\,\text{precip}_{t+k} > 0$
52. `precip_sum_next_24h` (zero-inflated regression)


In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet", "csv", "json", "dtype-datetime", "rolling_window", "strings", "temporal", "rank", "abs", "log", "trigonometry", "round_series", "cum_agg", "is_in"] }
:dep ndarray = { version = "0.16", features = ["serde"] }
:dep chrono = { version = "0.4", features = ["serde"] }
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep statrs = "0.18"

In [2]:
use polars::prelude::*;
use std::f64::consts::PI;
use std::fs::File;
use chrono::{NaiveDate, NaiveDateTime, Datelike, Timelike};

println!("Dependencies loaded.");

Dependencies loaded.


---
## 1. Load the raw data from Notebook 01

In [3]:
let parquet_path = "../data/raw/weather_sample_2024_01.parquet";
let df = LazyFrame::scan_parquet(parquet_path, Default::default())
    .expect("scan parquet")
    .collect()
    .expect("collect");

println!("Loaded: {} rows x {} columns", df.height(), df.width());
println!("Cities: {:?}",
    df.column("city").unwrap().str().unwrap().into_iter()
      .filter_map(|s| s.map(String::from))
      .collect::<std::collections::HashSet<_>>());

Loaded: 30912 rows x 23 columns


Cities: {"Los Angeles", "Campinas", "New York", "Oslo", "Tokyo", "Shanghai", "Sao Paulo", "Berlin", "Nanjing", "London", "Dubai", "Rio de Janeiro", "Sao Jose dos Campos", "Chongqing"}


---
## 2. Structural cleanup

The EDA showed that (i) `apparent_temperature` is collinear with
`temperature_2m` ($\rho = 0.989$) and (ii) 2 observations satisfy
$G < |U|$ due to API rounding. We apply:

1. **Drop** `apparent_temperature` — redundant information;
2. `gust_clipped = max(G, |U|)` — fixes the small inconsistency without losing data;
3. **Drop** `direct_radiation` — `shortwave_radiation` is already more general.

In [4]:
let df = df.lazy()
    .with_column(
        when(col("windgusts_10m").lt(col("windspeed_10m")))
            .then(col("windspeed_10m"))
            .otherwise(col("windgusts_10m"))
            .alias("windgusts_10m")
    )
    .drop(["apparent_temperature", "direct_radiation"])
    .collect().unwrap();

println!("After cleanup: {} x {}", df.height(), df.width());

After cleanup: 30912 x 21


---
## 3. Per-city imputation (no cross-station leakage)

Forward/backward fills must respect city boundaries: otherwise a null at
the start of the Tokyo block would be filled with the last value from
Shanghai. We use `fill_null(...).over("city")`.

The Nb01 EDA found an empty null inventory, so this imputation is
defensive — it protects the pipeline when the dataset is expanded later.

In [5]:
let numeric_cols = [
    "temperature_2m", "dewpoint_2m",
    "precipitation", "rain", "snowfall",
    "windspeed_10m", "windgusts_10m", "winddirection_10m",
    "pressure_msl", "surface_pressure", "cloudcover",
    "shortwave_radiation", "relativehumidity_2m",
];

// Sort by (city, timestamp) once so forward-fill respects time order.
let df = df.lazy()
    .sort(["city", "timestamp"], Default::default())
    .with_columns(
        numeric_cols.iter().map(|c|
            col(*c).forward_fill(None).backward_fill(None).over([col("city")])
        ).collect::<Vec<_>>()
    )
    // Accumulative variables should default to 0 when still null.
    .with_columns([
        col("precipitation").fill_null(lit(0.0)),
        col("rain").fill_null(lit(0.0)),
        col("snowfall").fill_null(lit(0.0)),
        col("shortwave_radiation").fill_null(lit(0.0)),
        col("weathercode").fill_null(lit(0i64)),
    ])
    .collect().unwrap();

let remaining: usize = df.get_columns().iter().map(|c| c.null_count()).sum();
println!("Nulls remaining after imputation: {}", remaining);

Nulls remaining after imputation: 0


---
## 4. Physical clipping

First-principles limits. Since the EDA already showed the data is well
within plausible ranges, this only affects extremes introduced by
imputation.

In [6]:
fn clip(name: &str, lo: f64, hi: f64) -> Expr {
    when(col(name).lt(lit(lo))).then(lit(lo))
        .when(col(name).gt(lit(hi))).then(lit(hi))
        .otherwise(col(name)).alias(name)
}

let df = df.lazy()
    .with_columns([
        clip("temperature_2m",      -60.0,  60.0),
        clip("dewpoint_2m",         -60.0,  40.0),
        clip("relativehumidity_2m",  0.0,  100.0),
        clip("windspeed_10m",        0.0,  300.0),
        clip("windgusts_10m",        0.0,  400.0),
        clip("pressure_msl",       870.0, 1084.0),
        clip("cloudcover",           0.0,  100.0),
        clip("precipitation",        0.0,  500.0),
        clip("shortwave_radiation",  0.0, 1500.0),
    ])
    .collect().unwrap();

println!("Physical clipping applied.");

Physical clipping applied.


---
## 5. Temporal parsing via `chrono::NaiveDate`

The original Notebook 02 had **two bugs**:

1. **Day-of-week** was computed with an incorrect heuristic: for
   2024-01-01 (Monday, code 0) it returned 4.
2. **Day-of-year** summed cumulative month lengths **without accounting
   for leap years** — in 2024 (a leap year) every date after February
   was off by one day.

The robust fix: parse each timestamp into `NaiveDateTime` and use
`year`, `month`, `day`, `hour`, `weekday`, `ordinal` from `chrono`. Cost:
$O(n)$ once — avoids reimplementing Zeller by hand.

In [7]:
// Parse "YYYY-MM-DDTHH:MM" into calendar components.
// We collect timestamps as Vec<String> (owned) so we don't hold a borrow
// on `df` while we add columns to it later.
let timestamps: Vec<String> = df.column("timestamp").unwrap().str().unwrap()
    .into_iter().map(|x| x.unwrap_or("1970-01-01T00:00").to_string()).collect();

let mut years   : Vec<i32> = Vec::with_capacity(timestamps.len());
let mut months  : Vec<i32> = Vec::with_capacity(timestamps.len());
let mut days    : Vec<i32> = Vec::with_capacity(timestamps.len());
let mut hours   : Vec<i32> = Vec::with_capacity(timestamps.len());
let mut dows    : Vec<i32> = Vec::with_capacity(timestamps.len());  // 0=Mon ... 6=Sun
let mut doys    : Vec<i32> = Vec::with_capacity(timestamps.len());  // 1..366

for ts in &timestamps {
    let dt = NaiveDateTime::parse_from_str(ts.as_str(), "%Y-%m-%dT%H:%M")
        .unwrap_or_else(|_| NaiveDateTime::parse_from_str("1970-01-01T00:00", "%Y-%m-%dT%H:%M").unwrap());
    years.push(dt.year());
    months.push(dt.month() as i32);
    days.push(dt.day() as i32);
    hours.push(dt.hour() as i32);
    dows.push(dt.weekday().num_days_from_monday() as i32);
    doys.push(dt.ordinal() as i32);
}

let mut df = df;
df.with_column(Series::new("year".into(),  years)).unwrap();
df.with_column(Series::new("month".into(), months)).unwrap();
df.with_column(Series::new("day".into(),   days)).unwrap();
df.with_column(Series::new("hour".into(),  hours)).unwrap();
df.with_column(Series::new("day_of_week".into(), dows)).unwrap();
df.with_column(Series::new("day_of_year".into(), doys)).unwrap();

// Sanity check: 2024-01-01 must have dow=0 (Monday).
let sample = df.clone().lazy()
    .filter(col("timestamp").eq(lit("2024-01-01T00:00")))
    .select([col("city"), col("day_of_week"), col("day_of_year")])
    .limit(3).collect().unwrap();
println!("Sanity check (2024-01-01 -> dow=0):");
println!("{}", sample);

Sanity check (2024-01-01 -> dow=0):


shape: (3, 3)


┌───────────┬─────────────┬─────────────┐


│ city      ┆ day_of_week ┆ day_of_year │


│ ---       ┆ ---         ┆ ---         │


...

╞═══════════╪═════════════╪═════════════╡


│ Berlin    ┆ 0           ┆ 1           │


│ Campinas  ┆ 0           ┆ 1           │


│ Chongqing ┆ 0           ┆ 1           │


└───────────┴─────────────┴─────────────┘


---
## 6. Cyclic encoding

Cyclical variables (hour-of-day, day-of-week, month, day-of-year) would
violate continuity assumptions if encoded as integers: 23h and 0h are
"adjacent" yet 23 units apart. We project them onto the unit circle:
$$
\big(\sin(2\pi k/K),\;\cos(2\pi k/K)\big).
$$

In [8]:
// Polars 0.46 exposes `.sin()` / `.cos()` on `Expr` once the
// `trigonometry` feature is enabled.

let df = df.lazy()
    .with_columns([
        // hour in [0, 23]
        ((col("hour").cast(DataType::Float64) * lit(2.0 * PI / 24.0)).sin()).alias("hour_sin"),
        ((col("hour").cast(DataType::Float64) * lit(2.0 * PI / 24.0)).cos()).alias("hour_cos"),
        // day of week in [0, 6]
        ((col("day_of_week").cast(DataType::Float64) * lit(2.0 * PI / 7.0)).sin()).alias("dow_sin"),
        ((col("day_of_week").cast(DataType::Float64) * lit(2.0 * PI / 7.0)).cos()).alias("dow_cos"),
        // month in [1, 12]
        (((col("month").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 12.0)).sin()).alias("month_sin"),
        (((col("month").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 12.0)).cos()).alias("month_cos"),
        // day of year in [1, 366]
        (((col("day_of_year").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 366.0)).sin()).alias("doy_sin"),
        (((col("day_of_year").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 366.0)).cos()).alias("doy_cos"),
    ])
    .collect().unwrap();

println!("Cyclic encoding applied.");

Cyclic encoding applied.


---
## 7. Water-vapour thermodynamics (Magnus-Tetens / Bolton 1980)

Bolton (1980) proposed a compact form for the saturation vapor pressure
used by virtually every modern NWP model:
$$
e_s(T) = 6.112 \cdot \exp\!\left(\frac{17.67\,T}{T + 243.5}\right)\quad\text{[hPa]},\quad T\ \text{in °C}.
$$
The actual vapor pressure follows by substituting $T \to T_d$. From there we derive:
- **Mixing ratio** $r = \varepsilon e / (P - e)$, with $\varepsilon = 0.622$
- **Specific humidity** $q = r/(1+r)$
- **VPD** $= e_s(T) - e_s(T_d)$ (vapor-pressure deficit)

VPD is one of the strongest predictors of evapotranspiration and —
indirectly — of rain onset, because it captures how "thirsty" the air is.

In [9]:
let df = df.lazy()
    .with_columns([
        // e_s(T)
        (lit(6.112) * (lit(17.67) * col("temperature_2m")
            / (col("temperature_2m") + lit(243.5))).exp()).alias("e_sat_T"),
        // e_s(T_d) = e_actual
        (lit(6.112) * (lit(17.67) * col("dewpoint_2m")
            / (col("dewpoint_2m") + lit(243.5))).exp()).alias("e_actual"),
    ])
    .with_columns([
        // VPD = e_s(T) - e_actual
        (col("e_sat_T") - col("e_actual")).alias("vpd"),
        // mixing ratio r
        (lit(0.622) * col("e_actual") / (col("pressure_msl") - col("e_actual"))).alias("mixing_ratio"),
    ])
    .with_columns([
        // specific humidity q = r/(1+r)
        (col("mixing_ratio") / (lit(1.0) + col("mixing_ratio"))).alias("specific_humidity"),
        // dewpoint depression
        (col("temperature_2m") - col("dewpoint_2m")).alias("dewpoint_depression"),
    ])
    .with_columns([
        // LCL approximation (Stull 1988): z_LCL ~ 125 m per 1 °C of (T-Td)
        (lit(125.0) * col("dewpoint_depression")).alias("lcl_height_m"),
    ])
    .collect().unwrap();

println!("Thermodynamic features created: e_sat_T, e_actual, vpd, mixing_ratio,");
println!("                                specific_humidity, dewpoint_depression, lcl_height_m");

// Quick check: VPD should never be negative (e_actual <= e_sat_T always).
let neg_vpd = df.clone().lazy().filter(col("vpd").lt(lit(-1e-3))).collect().unwrap().height();
println!("Negative VPD (should be 0): {}", neg_vpd);

Thermodynamic features created: e_sat_T, e_actual, vpd, mixing_ratio,


                                specific_humidity, dewpoint_depression, lcl_height_m


Negative VPD (should be 0): 0


---
## 8. Wind-vector decomposition

Wind direction $\theta$ is in degrees, measured clockwise from North
(meteorological convention: 0° = wind coming from the North). For linear
models we project it onto Cartesian components:
$$
u = -|U|\sin(\theta\,\pi/180),\quad v = -|U|\cos(\theta\,\pi/180).
$$
The negative sign follows the convention that $u>0$ means wind *blowing
east* (i.e. wind from the west).

We also add `gust_excess = G - |U|`, which is $\ge 0$ by construction
after the clipping in Section 2. The excess captures turbulence without
repeating the information already in $|U|$.

In [10]:
let df = df.lazy()
    .with_columns([
        (lit(-1.0) * col("windspeed_10m")
            * (col("winddirection_10m") * lit(PI / 180.0)).sin()).alias("u_wind"),
        (lit(-1.0) * col("windspeed_10m")
            * (col("winddirection_10m") * lit(PI / 180.0)).cos()).alias("v_wind"),
        (col("windgusts_10m") - col("windspeed_10m")).alias("gust_excess"),
        // log(1 + |U|) -- fixes the positive skew identified in the EDA
        ((col("windspeed_10m") + lit(1.0)).log(std::f64::consts::E)).alias("log_windspeed"),
    ])
    .collect().unwrap();

println!("Wind-vector features created: u_wind, v_wind, gust_excess, log_windspeed");

Wind-vector features created: u_wind, v_wind, gust_excess, log_windspeed


---
## 9. Solar geometry — declination, zenith angle, clearness index

The bimodal shortwave radiation observed in the EDA is purely geometric:
$S^{\downarrow}\approx 0$ at night and $S^{\downarrow}\sim 1000$ W/m²
at noon under clear sky. This explains the weak correlation with
`cloudcover` ($\rho = -0.16$): hour-of-day dominates the signal.

We build the **clearness index** $\tau = S^{\downarrow}/S_{clear}$,
where $S_{clear}$ is the radiation one would observe under a perfectly
clear sky. Simple approximation:
$$
S_{clear}(\phi, doy, h) = S_0 \cdot \max(0, \cos z),\quad S_0 = 1361\ \text{W m}^{-2},
$$
$$
\cos z = \sin\phi\sin\delta + \cos\phi\cos\delta\cos H,
$$
$$
\delta = 23.45°\sin\!\left(\frac{2\pi(284 + doy)}{365}\right),\quad H = 15°\cdot(h - 12).
$$

$\tau$ should sit in $[0, 1]$ with small overshoots from aerosol/raman
scattering. We expect $\rho(\tau, N) \ll -0.5$, capturing the real
cloud effect on solar transmission — which was invisible in raw
$S^{\downarrow}$.

In [11]:
let lat   = col("latitude") * lit(PI / 180.0);
let doy   = col("day_of_year").cast(DataType::Float64);
let hr    = col("hour").cast(DataType::Float64);

// Solar declination delta (in radians)
let decl = (lit(2.0 * PI) * (lit(284.0) + doy.clone()) / lit(365.0)).sin()
    * lit(23.45 * PI / 180.0);

// Hour angle H (radians): H=0 at noon, +/- pi at midnight
let hour_angle = (hr - lit(12.0)) * lit(15.0 * PI / 180.0);

// cos z = sin phi sin delta + cos phi cos delta cos H
let cos_zenith = (lat.clone().sin() * decl.clone().sin())
    + (lat.cos() * decl.cos() * hour_angle.cos());

let s0 = 1361.0_f64;

let df = df.lazy()
    .with_columns([
        cos_zenith.clone().alias("cos_solar_zenith"),
    ])
    .with_columns([
        // Clear-sky: clip at the horizon (cos z < 0 -> 0)
        (when(col("cos_solar_zenith").lt(lit(0.0))).then(lit(0.0))
            .otherwise(col("cos_solar_zenith") * lit(s0)))
            .alias("clear_sky_radiation"),
        // is_daytime flag
        (col("cos_solar_zenith").gt(lit(0.0))).alias("is_daytime"),
    ])
    .with_columns([
        // clearness index = S_obs / max(S_clear, 1)
        (col("shortwave_radiation")
            / when(col("clear_sky_radiation").lt(lit(1.0))).then(lit(1.0))
                .otherwise(col("clear_sky_radiation"))
        ).alias("clearness_index_raw"),
    ])
    .with_columns([
        // Clip tau to [0, 1.5] -- overshoots > 1 are physically possible (cloud edge)
        when(col("clearness_index_raw").lt(lit(0.0))).then(lit(0.0))
            .when(col("clearness_index_raw").gt(lit(1.5))).then(lit(1.5))
            .otherwise(col("clearness_index_raw"))
            .alias("clearness_index"),
    ])
    .drop(["clearness_index_raw"])
    .collect().unwrap();

// Validation: rho(clearness, cloudcover) should be much more negative than rho(S, cloud).
fn pearson(a: &[f64], b: &[f64]) -> f64 {
    let n = a.len() as f64;
    let ma = a.iter().sum::<f64>() / n;
    let mb = b.iter().sum::<f64>() / n;
    let mut cov = 0.0; let mut va = 0.0; let mut vb = 0.0;
    for (x, y) in a.iter().zip(b.iter()) {
        let dx = x - ma; let dy = y - mb;
        cov += dx*dy; va += dx*dx; vb += dy*dy;
    }
    if va*vb < 1e-18 { 0.0 } else { cov / (va*vb).sqrt() }
}

fn col_to_vec_f64(df: &DataFrame, c: &str) -> Vec<f64> {
    df.column(c).unwrap().f64().unwrap().into_iter().filter_map(|x| x).collect()
}
let only_day: Vec<usize> = df.column("is_daytime").unwrap().bool().unwrap()
    .into_iter().enumerate()
    .filter_map(|(i, b): (usize, Option<bool>)| if b == Some(true) { Some(i) } else { None })
    .collect();
let tau_all = col_to_vec_f64(&df, "clearness_index");
let cc_all  = col_to_vec_f64(&df, "cloudcover");
let tau_day: Vec<f64> = only_day.iter().map(|&i| tau_all[i]).collect();
let cc_day:  Vec<f64> = only_day.iter().map(|&i| cc_all[i]).collect();
println!("rho(clearness_index, cloudcover) [daytime only] = {:.4}", pearson(&tau_day, &cc_day));
println!("(compare with rho(S, cloud) = -0.16 from the EDA)");

rho(clearness_index, cloudcover) [daytime only] = -0.3508


(compare with rho(S, cloud) = -0.16 from the EDA)


---
## 10. Multi-scale lags

The Nb01 autocorrelation analysis showed:
- $\rho(1) \approx 0.99$ universal → **lag-1 essential**
- $\rho(24) > 0.83$ universal → **lag-24 essential**
- $\rho(12)$ ranges from $\approx 0$ (Brazilian cities) to $0.89$
  (Oslo, Shanghai) → lag-12 helps *only* in continental climates

We build lags over $\{1, 3, 6, 12, 24, 48\}$ h for temperature,
dewpoint, pressure, wind, and precipitation — chosen to span the typical
hourly and diurnal variability.

**Critical**: every shift is scoped `.over("city")` so row 0 of the next
city does not inherit the last value of the previous one.

In [12]:
// Re-sort to guarantee that subsequent .over() shifts are time-ordered.
let df = df.lazy().sort(["city", "timestamp"], Default::default()).collect().unwrap();

let lag_specs: Vec<(&str, &[i64])> = vec![
    ("temperature_2m",     &[1, 3, 6, 12, 24, 48]),
    ("dewpoint_2m",        &[1, 6, 24]),
    ("pressure_msl",       &[1, 3, 6, 12, 24]),
    ("windspeed_10m",      &[1, 6, 24]),
    ("relativehumidity_2m",&[1, 6, 24]),
    ("precipitation",      &[1, 3, 6]),
];

let mut lag_exprs: Vec<Expr> = Vec::new();
for (var, ks) in &lag_specs {
    for k in *ks {
        let alias = format!("{}_lag{}h", short_name(var), k);
        lag_exprs.push(col(*var).shift(lit(*k as i64)).over([col("city")]).alias(alias));
    }
}

fn short_name(v: &str) -> &'static str {
    match v {
        "temperature_2m"      => "temp",
        "dewpoint_2m"         => "td",
        "pressure_msl"        => "pres",
        "windspeed_10m"       => "wind",
        "relativehumidity_2m" => "rh",
        "precipitation"       => "precip",
        _ => "x",
    }
}

let df = df.lazy().with_columns(lag_exprs).collect().unwrap();
println!("Lags created: {} columns added.",
         lag_specs.iter().map(|(_, ks)| ks.len()).sum::<usize>());

Lags created: 23 columns added.


---
## 11. Gradients (tendencies)

Pressure tendency is the single strongest predictor of hourly rain
(Bjerknes 1898 — the very first numerical weather model). Drops of
$\Delta P / \Delta t > 1$ hPa in 3 h are associated with active fronts.

We compute $\Delta X_k = X_t - X_{t-k}$ for $k \in \{1, 3, 6\}$ h.

In [13]:
let df = df.lazy()
    .with_columns([
        (col("temperature_2m")    - col("temp_lag1h")).alias("temp_change_1h"),
        (col("temperature_2m")    - col("temp_lag3h")).alias("temp_change_3h"),
        (col("temperature_2m")    - col("temp_lag6h")).alias("temp_change_6h"),
        (col("temperature_2m")    - col("temp_lag24h")).alias("temp_change_24h"),
        (col("pressure_msl")      - col("pres_lag1h")).alias("pressure_change_1h"),
        (col("pressure_msl")      - col("pres_lag3h")).alias("pressure_change_3h"),
        (col("pressure_msl")      - col("pres_lag6h")).alias("pressure_change_6h"),
        (col("pressure_msl")      - col("pres_lag24h")).alias("pressure_change_24h"),
        (col("dewpoint_2m")       - col("td_lag1h")).alias("dewpoint_change_1h"),
        (col("dewpoint_2m")       - col("td_lag6h")).alias("dewpoint_change_6h"),
        (col("relativehumidity_2m") - col("rh_lag1h")).alias("rh_change_1h"),
        (col("windspeed_10m")     - col("wind_lag1h")).alias("wind_change_1h"),
    ])
    .collect().unwrap();

println!("Gradients (12 features) computed.");

Gradients (12 features) computed.


---
## 12. Rolling-window statistics (24 h)

The EDA showed the diurnal range spans 6 °C (Rio) to 12 °C (LA). Only
rolling windows capture this information. We compute `mean`, `std`,
`min`, `max`, and `range` over a 24 h window — every one scoped
`.over("city")`.

In [14]:
// Polars 0.46 exposes `rolling_mean / std / min / max / sum` on `Expr`
// when the `rolling_window` feature is enabled. Each takes a
// RollingOptionsFixedWindow. Combined with `.over("city")` we guarantee
// each city has its own rolling without leakage across stations.

let opts = RollingOptionsFixedWindow {
    window_size: 24,
    min_periods: 1,
    weights: None,
    center: false,
    fn_params: None,
};

let df = df.lazy()
    .sort(["city", "timestamp"], Default::default())
    .with_columns([
        col("temperature_2m").rolling_mean(opts.clone()).over([col("city")]).alias("temp_roll24_mean"),
        col("temperature_2m").rolling_std(opts.clone()).over([col("city")]).alias("temp_roll24_std"),
        col("temperature_2m").rolling_min(opts.clone()).over([col("city")]).alias("temp_roll24_min"),
        col("temperature_2m").rolling_max(opts.clone()).over([col("city")]).alias("temp_roll24_max"),
        col("pressure_msl").rolling_min(opts.clone()).over([col("city")]).alias("pres_roll24_min"),
        col("pressure_msl").rolling_max(opts.clone()).over([col("city")]).alias("pres_roll24_max"),
        col("precipitation").rolling_sum(opts.clone()).over([col("city")]).alias("precip_roll24_sum"),
    ])
    .with_column((col("temp_roll24_max") - col("temp_roll24_min")).alias("temp_diurnal_range_24h"))
    .collect().unwrap();

println!("Rolling features (8 cols) created.");

Rolling features (8 cols) created.


---
## 13. Physical interactions

Linear models do not capture interactions on their own. We add three
products with physical interpretation:

- $T \cdot RH$ — crude heat-index proxy; high humidity + high T
  ⇒ discomfort and low evaporation.
- $|U| \cdot \Delta T_d$ — latent-heat transfer flux; wind + dry air
  accelerate surface evaporation.
- $|U| \cdot VPD$ — vapor-diffusion rate; governs how fast the
  atmosphere approaches saturation.

In [15]:
let df = df.lazy()
    .with_columns([
        (col("temperature_2m") * col("relativehumidity_2m") / lit(100.0)).alias("interact_t_rh"),
        (col("windspeed_10m") * col("dewpoint_depression")).alias("interact_wind_ddep"),
        (col("windspeed_10m") * col("vpd")).alias("interact_wind_vpd"),
    ])
    .collect().unwrap();

println!("Interaction features created.");

Interaction features created.


---
## 14. Geographic + Köppen one-hot

The EDA showed that cities cluster by Köppen type. We encode that as
one-hot (8 zones in the dataset). We also add $|\phi|$ (distance from
the equator), which is more informative than the raw $\phi$.

In [16]:
let zones = ["Aw", "BWh", "Cfa", "Cfb", "Csb", "Cwa", "Dfa", "Dfb"];

let mut zone_exprs: Vec<Expr> = vec![
    (col("latitude").abs()).alias("abs_latitude"),
];
for z in &zones {
    zone_exprs.push(
        when(col("climate_zone").eq(lit(*z)))
            .then(lit(1i32))
            .otherwise(lit(0i32))
            .alias(format!("climate_{}", z))
    );
}

let df = df.lazy().with_columns(zone_exprs).collect().unwrap();
println!("Geographic + Köppen one-hot created ({} cols).", 1 + zones.len());

Geographic + Köppen one-hot created (9 cols).


---
## 15. Leakage-free targets

The original Notebook 02 defined `will_rain = precipitation > 0` — this
**leaks the target into the features**, because the feature
`precipitation` is one of the model inputs. Random Forest reached 100%
accuracy trivially (it just had to read `precipitation` to predict
`will_rain`).

Here we define *future targets*:

- `temp_next_24h, temp_next_48h, temp_next_72h` — negative shift of $T$
- `will_rain_next_24h` — rain falls in at least one of the next 24 h
- `precip_sum_next_24h` — total accumulation over the next 24 h

For `will_rain_next_24h` we apply `shift(-24)` to the 24 h rolling sum
of precipitation (`precip_roll24_sum`), producing "sum of the next 24 h"
— then binarize.

**Important**: features `precipitation`, `rain`, `precip_lagXh`,
`precip_roll24_sum` remain available to the model, but since they count
only the **past** ($\le t$), not the future, there is no leakage — the
model still has to generalize to predict what will happen.

In [17]:
// WMO 4677 -> 6 classes
fn wmo_to_condition() -> Expr {
    when(col("weathercode").eq(lit(0i64)).or(col("weathercode").eq(lit(1i64))))
        .then(lit(0i64))  // Clear
    .when(col("weathercode").eq(lit(2i64)).or(col("weathercode").eq(lit(3i64))))
        .then(lit(1i64))  // Cloudy
    .when(col("weathercode").eq(lit(45i64)).or(col("weathercode").eq(lit(48i64))))
        .then(lit(2i64))  // Foggy
    .when(col("weathercode").gt_eq(lit(51i64)).and(col("weathercode").lt_eq(lit(67i64))))
        .then(lit(3i64))  // Rainy
    .when(col("weathercode").gt_eq(lit(80i64)).and(col("weathercode").lt_eq(lit(82i64))))
        .then(lit(3i64))
    .when(col("weathercode").gt_eq(lit(71i64)).and(col("weathercode").lt_eq(lit(77i64))))
        .then(lit(4i64))  // Snowy
    .when(col("weathercode").eq(lit(85i64)).or(col("weathercode").eq(lit(86i64))))
        .then(lit(4i64))
    .when(col("weathercode").gt_eq(lit(95i64)))
        .then(lit(5i64))  // Stormy
    .otherwise(lit(1i64))  // Default -> Cloudy (modal class)
}

let df = df.lazy()
    .sort(["city", "timestamp"], Default::default())
    .with_columns([
        col("temperature_2m").shift(lit(-24i64)).over([col("city")]).alias("temp_next_24h"),
        col("temperature_2m").shift(lit(-48i64)).over([col("city")]).alias("temp_next_48h"),
        col("temperature_2m").shift(lit(-72i64)).over([col("city")]).alias("temp_next_72h"),
        // sum over the next 24h: shift(-24) of the rolling 24h sum
        col("precip_roll24_sum").shift(lit(-24i64)).over([col("city")]).alias("precip_sum_next_24h"),
        wmo_to_condition().alias("weather_condition"),
    ])
    .with_column(
        (col("precip_sum_next_24h").gt(lit(0.0)))
            .cast(DataType::Int64)
            .alias("will_rain_next_24h")
    )
    .collect().unwrap();

// Sanity: distribution of the target
let dist = df.clone().lazy()
    .group_by([col("will_rain_next_24h")])
    .agg([col("city").count().alias("n")])
    .sort(["will_rain_next_24h"], Default::default())
    .collect().unwrap();
println!("will_rain_next_24h distribution:");
println!("{}", dist);

let cond = df.clone().lazy()
    .group_by([col("weather_condition")])
    .agg([col("city").count().alias("n")])
    .sort(["weather_condition"], Default::default())
    .collect().unwrap();
println!("weather_condition distribution:");
println!("{}", cond);

will_rain_next_24h distribution:


shape: (3, 2)


┌────────────────────┬───────┐


│ will_rain_next_24h ┆ n     │


│ ---                ┆ ---   │


...

│ 0                 ┆ 12126 │


│ 1                 ┆ 13765 │


│ 3                 ┆ 4738  │


│ 4                 ┆ 283   │


└───────────────────┴───────┘


---
## 16. Skewness remedies

We apply `log1p` to variables with strong positive skew detected in
Nb01 (precipitation g₁=+11, windspeed g₁=+1.26). The transform
$y = \log(1 + x)$ keeps zero intact and damps long tails.

In [18]:
let df = df.lazy()
    .with_columns([
        (col("precipitation") + lit(1.0)).log(std::f64::consts::E).alias("log1p_precipitation"),
        (col("windspeed_10m") + lit(1.0)).log(std::f64::consts::E).alias("log1p_windspeed"),
        // log version of the zero-inflated target
        (col("precip_sum_next_24h") + lit(1.0)).log(std::f64::consts::E).alias("log1p_precip_next_24h"),
    ])
    .collect().unwrap();

println!("log1p transforms applied.");

log1p transforms applied.


---
## 17. Final feature inventory

In [19]:
println!("=== FINAL INVENTORY ===");
println!("Total columns: {}", df.width());
println!("Total rows:    {}", df.height());
println!();

let cats: std::collections::BTreeMap<&str, Vec<String>> = {
    let mut m: std::collections::BTreeMap<&str, Vec<String>> = std::collections::BTreeMap::new();
    for name in df.get_column_names() {
        let n = name.to_string();
        let cat = if ["city","country_code","climate_zone","timestamp"].contains(&n.as_str()) { "metadata" }
                  else if ["latitude","longitude","elevation_m","abs_latitude"].contains(&n.as_str()) { "geo" }
                  else if n.starts_with("climate_") { "climate_onehot" }
                  else if ["year","month","day","hour","day_of_week","day_of_year"].contains(&n.as_str()) { "time_raw" }
                  else if n.ends_with("_sin") || n.ends_with("_cos") { "time_cyclic" }
                  else if n.contains("_lag") { "lag" }
                  else if n.contains("_roll") { "rolling" }
                  else if n.contains("_change") { "gradient" }
                  else if n.starts_with("interact_") { "interaction" }
                  else if ["e_sat_T","e_actual","vpd","mixing_ratio","specific_humidity",
                            "dewpoint_depression","lcl_height_m"].contains(&n.as_str()) { "thermo" }
                  else if ["u_wind","v_wind","gust_excess","log_windspeed"].contains(&n.as_str()) { "wind_vec" }
                  else if ["cos_solar_zenith","clear_sky_radiation","is_daytime","clearness_index"].contains(&n.as_str()) { "solar" }
                  else if ["log1p_precipitation","log1p_windspeed","log1p_precip_next_24h"].contains(&n.as_str()) { "log_transform" }
                  else if n.starts_with("temp_next_") || n.starts_with("will_rain_") || n.starts_with("precip_sum_next_") || n == "weather_condition" { "target" }
                  else { "raw_obs" };
        m.entry(cat).or_default().push(n);
    }
    m
};

for (k, v) in &cats {
    println!("  {:<16} ({:>3}): {}", k, v.len(),
             if v.len() < 8 { v.join(", ") } else { format!("{}...", v[..5].join(", ")) });
}

=== FINAL INVENTORY ===


Total columns: 114


Total rows:    30912


  climate_onehot   (  8): climate_Aw, climate_BWh, climate_Cfa, climate_Cfb, climate_Csb...


  geo              (  4): latitude, longitude, elevation_m, abs_latitude


...

  thermo           (  7): e_sat_T, e_actual, vpd, mixing_ratio, specific_humidity, dewpoint_depression, lcl_height_m


  time_cyclic      (  8): hour_sin, hour_cos, dow_sin, dow_cos, month_sin...


  time_raw         (  6): year, month, day, hour, day_of_week, day_of_year


  wind_vec         (  4): u_wind, v_wind, gust_excess, log_windspeed


()

---
## 18. Per-city temporal split

For the multi-month sample we do a **per-city temporal split**:
- Train: days 1–20 of each fetched month
- Val:   days 21–25
- Test:  days 26–31

This preserves the temporal structure (no training on the future) and
guarantees that every city shows up in all three splits.

In [20]:
let train_df = df.clone().lazy()
    .filter(col("day").lt_eq(lit(20i32)))
    .collect().unwrap();
let val_df = df.clone().lazy()
    .filter(col("day").gt(lit(20i32)).and(col("day").lt_eq(lit(25i32))))
    .collect().unwrap();
let test_df = df.clone().lazy()
    .filter(col("day").gt(lit(25i32)))
    .collect().unwrap();

println!("=== TEMPORAL SPLIT ===");
println!("Train: {:>5} ({:.1}%)", train_df.height(), 100.0 * train_df.height() as f64 / df.height() as f64);
println!("Val:   {:>5} ({:.1}%)", val_df.height(),   100.0 * val_df.height() as f64 / df.height() as f64);
println!("Test:  {:>5} ({:.1}%)", test_df.height(),  100.0 * test_df.height() as f64 / df.height() as f64);

// Target distribution in each split
for (name, split) in [("Train", &train_df), ("Val", &val_df), ("Test", &test_df)] {
    let total = split.height() as f64;
    if total == 0.0 { continue; }
    let rain = split.clone().lazy()
        .filter(col("will_rain_next_24h").eq(lit(1i64)))
        .collect().unwrap().height();
    println!("{:<6} -> will_rain_next_24h positive: {:>5} ({:.1}%)",
             name, rain, 100.0 * rain as f64 / total);
}

=== TEMPORAL SPLIT ===


Train: 20160 (65.2%)


Val:    5040 (16.3%)


Test:   5712 (18.5%)


Train  -> will_rain_next_24h positive: 14831 (73.6%)


Val    -> will_rain_next_24h positive:  3611 (71.6%)


Test   -> will_rain_next_24h positive:  3830 (67.1%)


()

---
## 19. Persistence

We save:
- `data/processed/weather_processed.parquet` — the full dataset
- `data/features/{train,val,test}.parquet` — splits ready for Nb03+

In [21]:
std::fs::create_dir_all("../data/processed").unwrap();
std::fs::create_dir_all("../data/features").unwrap();

let mut f = File::create("../data/processed/weather_processed.parquet").unwrap();
ParquetWriter::new(&mut f).finish(&mut df.clone()).unwrap();
println!("Saved ../data/processed/weather_processed.parquet");

let mut f = File::create("../data/features/train.parquet").unwrap();
ParquetWriter::new(&mut f).finish(&mut train_df.clone()).unwrap();
println!("Saved ../data/features/train.parquet ({} rows)", train_df.height());

let mut f = File::create("../data/features/val.parquet").unwrap();
ParquetWriter::new(&mut f).finish(&mut val_df.clone()).unwrap();
println!("Saved ../data/features/val.parquet ({} rows)", val_df.height());

let mut f = File::create("../data/features/test.parquet").unwrap();
ParquetWriter::new(&mut f).finish(&mut test_df.clone()).unwrap();
println!("Saved ../data/features/test.parquet ({} rows)", test_df.height());

Saved ../data/processed/weather_processed.parquet


Saved ../data/features/train.parquet (20160 rows)


Saved ../data/features/val.parquet (5040 rows)


Saved ../data/features/test.parquet (5712 rows)


---
## 20. Feature-engineering schema (contract for `src/features/`)

We export a machine-readable specification of every derived feature.
The Rust port must implement *exactly* these transformations, in this
order, with these exact column names. Any deviation fails the
equivalence test in Section 21.

In [22]:
let schema = serde_json::json!({
    "version": "1.0.0",
    "generator": "Notebook 02",
    "input_columns": [
        "city", "country_code", "climate_zone", "latitude", "longitude",
        "elevation_m", "timestamp", "temperature_2m", "dewpoint_2m",
        "precipitation", "rain", "snowfall",
        "windspeed_10m", "windgusts_10m", "winddirection_10m",
        "pressure_msl", "surface_pressure", "cloudcover",
        "shortwave_radiation", "relativehumidity_2m", "weathercode"
    ],
    "drop_columns": ["apparent_temperature", "direct_radiation"],
    "clip_pre_filter": {
        "windgusts_10m": "max(windgusts_10m, windspeed_10m)"
    },
    "physical_clips": {
        "temperature_2m":      [-60.0,  60.0],
        "dewpoint_2m":         [-60.0,  40.0],
        "relativehumidity_2m": [  0.0, 100.0],
        "windspeed_10m":       [  0.0, 300.0],
        "windgusts_10m":       [  0.0, 400.0],
        "pressure_msl":        [870.0,1084.0],
        "cloudcover":          [  0.0, 100.0],
        "precipitation":       [  0.0, 500.0],
        "shortwave_radiation": [  0.0,1500.0]
    },
    "thermodynamic_features": {
        "e_sat_T":             {"formula": "6.112 * exp(17.67 * T / (T + 243.5))", "units": "hPa"},
        "e_actual":            {"formula": "6.112 * exp(17.67 * Td / (Td + 243.5))", "units": "hPa"},
        "vpd":                 {"formula": "e_sat_T - e_actual", "units": "hPa"},
        "mixing_ratio":        {"formula": "0.622 * e_actual / (pressure_msl - e_actual)", "units": "kg/kg"},
        "specific_humidity":   {"formula": "mixing_ratio / (1 + mixing_ratio)", "units": "kg/kg"},
        "dewpoint_depression": {"formula": "T - Td", "units": "°C"},
        "lcl_height_m":        {"formula": "125.0 * dewpoint_depression", "units": "m"}
    },
    "wind_vector_features": {
        "u_wind":        "-windspeed_10m * sin(winddirection_10m * PI / 180)",
        "v_wind":        "-windspeed_10m * cos(winddirection_10m * PI / 180)",
        "gust_excess":   "windgusts_10m - windspeed_10m",
        "log_windspeed": "ln(windspeed_10m + 1)"
    },
    "solar_features": {
        "solar_constant_W_m2": 1361.0,
        "declination_deg":     "23.45 * sin(2 * PI * (284 + doy) / 365)",
        "hour_angle_deg":      "15 * (hour - 12)",
        "cos_solar_zenith":    "sin(phi)*sin(delta) + cos(phi)*cos(delta)*cos(H)",
        "clear_sky_radiation": "max(0, cos_solar_zenith) * 1361",
        "is_daytime":          "cos_solar_zenith > 0",
        "clearness_index":     "clip(shortwave_radiation / max(clear_sky_radiation, 1), 0, 1.5)"
    },
    "cyclic_encoding": {
        "hour":  ["hour_sin", "hour_cos"],
        "dow":   ["dow_sin",  "dow_cos"],
        "month": ["month_sin","month_cos"],
        "doy":   ["doy_sin",  "doy_cos"]
    },
    "lag_specifications": [
        {"variable": "temperature_2m",     "lags_hours": [1,3,6,12,24,48], "prefix": "temp_lag"},
        {"variable": "dewpoint_2m",        "lags_hours": [1,6,24],         "prefix": "td_lag"},
        {"variable": "pressure_msl",       "lags_hours": [1,3,6,12,24],    "prefix": "pres_lag"},
        {"variable": "windspeed_10m",      "lags_hours": [1,6,24],         "prefix": "wind_lag"},
        {"variable": "relativehumidity_2m","lags_hours": [1,6,24],         "prefix": "rh_lag"},
        {"variable": "precipitation",      "lags_hours": [1,3,6],          "prefix": "precip_lag"}
    ],
    "gradient_features": [
        "temp_change_1h", "temp_change_3h", "temp_change_6h", "temp_change_24h",
        "pressure_change_1h", "pressure_change_3h", "pressure_change_6h", "pressure_change_24h",
        "dewpoint_change_1h", "dewpoint_change_6h",
        "rh_change_1h", "wind_change_1h"
    ],
    "rolling_window": {
        "window_size_hours": 24,
        "min_periods": 1,
        "center": false,
        "features": [
            "temp_roll24_mean", "temp_roll24_std", "temp_roll24_min", "temp_roll24_max",
            "pres_roll24_min", "pres_roll24_max", "precip_roll24_sum", "temp_diurnal_range_24h"
        ]
    },
    "interaction_features": [
        {"name": "interact_t_rh",      "formula": "temperature_2m * relativehumidity_2m / 100"},
        {"name": "interact_wind_ddep", "formula": "windspeed_10m * dewpoint_depression"},
        {"name": "interact_wind_vpd",  "formula": "windspeed_10m * vpd"}
    ],
    "climate_zones_one_hot": ["Aw", "BWh", "Cfa", "Cfb", "Csb", "Cwa", "Dfa", "Dfb"],
    "log_transforms": [
        {"name": "log1p_precipitation",   "formula": "ln(precipitation + 1)"},
        {"name": "log1p_windspeed",       "formula": "ln(windspeed_10m + 1)"},
        {"name": "log1p_precip_next_24h", "formula": "ln(precip_sum_next_24h + 1)"}
    ],
    "targets": [
        {"name": "temp_next_24h",       "type": "regression", "formula": "temperature_2m.shift(-24).over(city)"},
        {"name": "temp_next_48h",       "type": "regression", "formula": "temperature_2m.shift(-48).over(city)"},
        {"name": "temp_next_72h",       "type": "regression", "formula": "temperature_2m.shift(-72).over(city)"},
        {"name": "precip_sum_next_24h", "type": "regression", "formula": "precip_roll24_sum.shift(-24).over(city)"},
        {"name": "will_rain_next_24h",  "type": "classification", "formula": "(precip_sum_next_24h > 0) as i64"},
        {"name": "weather_condition",   "type": "classification", "formula": "WMO 4677 mapping (see notebook)"}
    ],
    "splits": {
        "strategy": "temporal_per_city",
        "train":   "day in [1, 20]",
        "val":     "day in (20, 25]",
        "test":    "day in (25, 31]"
    },
    "n_columns_final": df.width(),
    "n_rows_full":     df.height()
});

std::fs::write("../models/features_schema.json",
    serde_json::to_string_pretty(&schema).unwrap()).unwrap();
println!("Saved ../models/features_schema.json");

Saved ../models/features_schema.json


---
## 21. Pipeline snapshot (50 rows for bit-exact equivalence tests)

We pick 50 rows deterministically (every `n/50`-th row) and save:
- the raw input values (the 21 columns read from the Parquet),
- the final engineered features for those same 50 rows.

The Rust port in `src/features/` will load this JSON, reconstruct the
transformation on the raw inputs, and assert the output matches
**within $10^{-9}$**. Any divergence fails the integration test.

In [23]:
// Deterministic sampling: every n/50-th row.
let n_rows = df.height();
let step = (n_rows / 50).max(1);
let sample_idx: Vec<usize> = (0..n_rows).step_by(step).take(50).collect();

// Which columns to include. We keep the raw inputs + every numeric derived feature.
let raw_cols = [
    "city", "climate_zone", "timestamp", "latitude", "longitude", "elevation_m",
    "temperature_2m", "dewpoint_2m", "precipitation", "rain", "snowfall",
    "windspeed_10m", "windgusts_10m", "winddirection_10m",
    "pressure_msl", "surface_pressure", "cloudcover",
    "shortwave_radiation", "relativehumidity_2m", "weathercode"
];

// Derived features to assert equivalence on (one per family).
let derived_cols = [
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "e_sat_T", "e_actual", "vpd", "mixing_ratio", "specific_humidity",
    "dewpoint_depression", "lcl_height_m",
    "u_wind", "v_wind", "gust_excess", "log_windspeed",
    "cos_solar_zenith", "clear_sky_radiation", "clearness_index",
    "temp_lag1h", "temp_lag24h", "pres_lag6h",
    "temp_change_1h", "pressure_change_6h",
    "temp_roll24_mean", "temp_roll24_std", "precip_roll24_sum", "temp_diurnal_range_24h",
    "interact_t_rh", "interact_wind_vpd",
    "temp_next_24h", "will_rain_next_24h"
];

fn val_at(df: &DataFrame, col_name: &str, i: usize) -> serde_json::Value {
    let col = df.column(col_name).unwrap();
    match col.dtype() {
        DataType::Float64 => {
            match col.f64().unwrap().get(i) {
                Some(v) if v.is_finite() => serde_json::json!(v),
                _ => serde_json::Value::Null,
            }
        }
        DataType::Int64 | DataType::Int32 => {
            let v = col.cast(&DataType::Int64).unwrap();
            match v.i64().unwrap().get(i) {
                Some(v) => serde_json::json!(v),
                _ => serde_json::Value::Null,
            }
        }
        DataType::Boolean => {
            match col.bool().unwrap().get(i) {
                Some(v) => serde_json::json!(v),
                _ => serde_json::Value::Null,
            }
        }
        DataType::String => {
            match col.str().unwrap().get(i) {
                Some(v) => serde_json::json!(v),
                _ => serde_json::Value::Null,
            }
        }
        _ => serde_json::Value::Null,
    }
}

let mut snapshot_rows: Vec<serde_json::Value> = Vec::new();
for &i in &sample_idx {
    let mut raw = serde_json::Map::new();
    for c in &raw_cols {
        if df.get_column_names().iter().any(|n| n.as_str() == *c) {
            raw.insert(c.to_string(), val_at(&df, c, i));
        }
    }
    let mut derived = serde_json::Map::new();
    for c in &derived_cols {
        if df.get_column_names().iter().any(|n| n.as_str() == *c) {
            derived.insert(c.to_string(), val_at(&df, c, i));
        }
    }
    snapshot_rows.push(serde_json::json!({
        "row_index": i,
        "raw_input": serde_json::Value::Object(raw),
        "expected_derived": serde_json::Value::Object(derived),
    }));
}

let snapshot = serde_json::json!({
    "version": "1.0.0",
    "generator": "Notebook 02",
    "tolerance_abs": 1e-9,
    "n_samples": sample_idx.len(),
    "samples": snapshot_rows,
});

std::fs::write("../models/pipeline_snapshot.json",
    serde_json::to_string_pretty(&snapshot).unwrap()).unwrap();
println!("Saved ../models/pipeline_snapshot.json ({} samples)", sample_idx.len());

Saved ../models/pipeline_snapshot.json (50 samples)


---
## 22. Summary

| category | n |
|---|---|
| Thermodynamics (Magnus-Tetens) | 7 |
| Wind vector | 4 |
| Solar geometry | 4 |
| Cyclic (sin/cos) | 8 |
| Multi-scale lags | 21 |
| Gradients | 12 |
| Rolling 24 h | 8 |
| Physical interactions | 3 |
| Geography + Köppen one-hot | 9 |
| Log transforms | 3 |
| **Leakage-free targets** | **6** |

New in this revision:

- `models/features_schema.json` — machine-readable contract
- `models/pipeline_snapshot.json` — 50 bit-exact regression samples

-> Next: **Notebook 03** — feature selection and model training, with
persistence baseline, regularization (Ridge/Lasso), and honest metrics
that account for class imbalance.

In [24]:
println!("\n{}", "=".repeat(60));
println!("Notebook 02 complete.");
println!("{}", "=".repeat(60));
println!("Next: Notebook 03 - Feature Selection & Model Training");

Notebook 02 complete.


Next: Notebook 03 - Feature Selection & Model Training
